In [1]:
%cd /content
!rm -rf SynBrain
!git clone https://github.com/ngoax/SynBrain

/content
Cloning into 'SynBrain'...
remote: Enumerating objects: 440, done.
remote: Counting objects: 100% (440/440), done.
remote: Compressing objects: 100% (333/333), done.
remote: Total 440 (delta 147), reused 366 (delta 91), pack-reused 0 (from 0)
Receiving objects: 100% (440/440), 31.41 MiB | 1.16 MiB/s, done.
Resolving deltas: 100% (147/147), done.


In [2]:
!ls SynBrain

configs		      download-sub-1.py  requirements.yml  SynBrain.png
create_dummy_data.py  LICENSE		 run_colab.ipynb
data		      README.md		 src


In [3]:
%cd /content/SynBrain
!pip install torchdiffeq einops pytorch-lightning omegaconf timm scipy scikit-learn seaborn wandb huggingface_hub webdataset braceexpand

/content/SynBrain
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 64.5 MB/s eta 0:00:00


In [4]:
# Generate dummy data and download checkpoints
!python create_dummy_data.py
!python download-sub-1.py

Creating train fMRI: (750, 15724)
  Saved. Size: 47.2 MB
Creating train CLIP: (250, 256, 1664)
  Saved. Size: 426.0 MB
Creating test fMRI: (90, 15724)
  Saved. Size: 5.7 MB
Creating test CLIP: (30, 256, 1664)
  Saved. Size: 51.1 MB

Dummy data created in: /content/SynBrain/dummy_data/subj01
Total files: 4

To train Stage 1 with this data, run:
  cd /content/SynBrain
  python src/vae/train_vae.py \
    --data_path /content/SynBrain/dummy_data \
    --save_path ./output \
    --subject '[1]' --valid-sub 1 --hour 1 \
    --batch_size 4 --num_epochs 5 --wandb_log False --plot_recon False
[SynBrain] Stage 1 checkpoint:
  [download] checkpoint/vae-nsd-s1-vs1-bs24-350/last.pth...
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
checkpoint/vae-nsd-s1-vs1-bs24-350/last.(…): 100% 940M/9

In [5]:
# Copy pre-trained checkpoint to where --resume expects it
import os
os.makedirs("output/train_logs/vae-nsd-s1-vs1-bs24-350", exist_ok=True)
!cp checkpoint/vae-nsd-s1-vs1-bs24-350/last.pth output/train_logs/vae-nsd-s1-vs1-bs24-350/last.pth
print("Checkpoint copied.")

Checkpoint copied.


In [6]:
!nvidia-smi

Mon Mar 16 15:54:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
!WANDB_MODE=disabled python src/vae/train_vae.py \
    --data_path ./dummy_data \
    --save_path ./output \
    --subject '[1]' \
    --valid-sub 1 \
    --hour 1 \
    --batch_size 4 \
    --num_epochs 3 \
    --base_lr 1e-4 \
    --resume

Using Layers: [2, 4, 4]
0.7967762 -0.783175
Train Voxel Sub1: (750, 15724)
Train Clip Sub1: (750, 256, 1664)
0.38842443 -0.40473795
Valid Voxel Sub1: (30, 15724)
Valid Clip Sub1: (30, 256, 1664)
Training batch size: 4
Dataset samples: fmri-750 feature-750
Testing batch size: 300
Dataset samples: fmri-30 feature-30

Done with Data preparations!
Using device: cuda, GPUs: 1
making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 256, 128, 128) = 4194304 dimensions.
making attention of type 'vanilla' with 512 in_channels
params of BrainAutoencoder
param counts:
78,311,937 total
78,311,937 trainable
=> resume checkpoint (iterations 12)
vae-nsd-s1-vs1-bs24-350 starting with epoch 14 / 3
0it [00:00, ?it/s]
